In [7]:
import pandas as pd
import numpy as np

from sklearn.linear_model import ElasticNet
from sklearn.model_selection import train_test_split

DATA_PATH = "../Opsium-challenge/Data/"

df = pd.read_csv(DATA_PATH + "customer_sku_demand_signals.csv")

print("Shape:", df.shape)
df.head()



Shape: (1200, 12)


,customer_id,sku_id,route,time_period,base_demand,promotion_flag,discount_percentage,sentiment_score,review_volume,sustainable_sku_flag,regulation_impact_score,eco_preference_index
0,CUST1,SKU1,DEL-FRA,2026-01-01,181.4,0,12,-0.016565,198,0,0.445833,0.099975
1,CUST1,SKU1,DEL-FRA,2026-01-02,186.9,0,12,0.133709,161,0,0.056412,0.721999
2,CUST1,SKU1,DEL-FRA,2026-01-03,229.3,1,25,-0.199221,170,0,0.524756,0.431945
3,CUST1,SKU1,DEL-FRA,2026-01-04,197.0,0,14,0.411853,197,0,0.456070,0.785176
4,CUST1,SKU1,DEL-FRA,2026-01-05,193.4,0,29,0.314234,140,1,0.680308,0.450499


In [8]:
df["promo_discount_effect"] = df["promotion_flag"] * df["discount_percentage"]
df["sentiment_strength"] = df["sentiment_score"] * np.log1p(df["review_volume"])
df["sustainability_alignment"] = (
    df["sustainable_sku_flag"] * df["eco_preference_index"]
)


In [10]:
np.random.seed(42)

df["observed_demand"] = (
    df["base_demand"]
    * (1 + 0.30 * df["promo_discount_effect"])
    * (1 + 0.20 * df["sentiment_strength"])
    * (1 + 0.15 * df["sustainability_alignment"])
    * (1 - 0.10 * df["regulation_impact_score"])
    * np.random.normal(1.0, 0.05, len(df))
)


In [11]:
features = [
    "base_demand",
    "promo_discount_effect",
    "sentiment_strength",
    "sustainability_alignment",
    "regulation_impact_score"
]

X = df[features]
y = df["observed_demand"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = ElasticNet(alpha=0.1, l1_ratio=0.5)
model.fit(X_train, y_train)


ElasticNet(alpha=0.1)

In [12]:
df["forecasted_demand"] = model.predict(X)

df["forecast_confidence"] = (
    1 - abs(df["forecasted_demand"] - df["observed_demand"]) / df["observed_demand"]
).clip(0, 1)


In [13]:
coefficients = pd.DataFrame({
    "Feature": features,
    "Impact": model.coef_
}).sort_values(by="Impact", ascending=False)

coefficients


,Feature,Impact
1,promo_discount_effect,69.536071
2,sentiment_strength,63.575970
3,sustainability_alignment,63.098282
0,base_demand,2.877319
4,regulation_impact_score,-34.259395


In [15]:
forecast_output = df[[
    "customer_id",
    "sku_id",
    "route",
    "time_period",
    "base_demand",
    "forecasted_demand",
    "forecast_confidence"
]]

forecast_output.to_csv(
    DATA_PATH + "Model_forecasted_demand_output.csv",
    index=False
)

forecast_output.head()


,customer_id,sku_id,route,time_period,base_demand,forecasted_demand,forecast_confidence
0,CUST1,SKU1,DEL-FRA,2026-01-01,181.4,175.217753,0.995899
1,CUST1,SKU1,DEL-FRA,2026-01-02,186.9,253.206883,0.792359
2,CUST1,SKU1,DEL-FRA,2026-01-03,229.3,1989.191211,0.687857
3,CUST1,SKU1,DEL-FRA,2026-01-04,197.0,363.795559,0.747558
4,CUST1,SKU1,DEL-FRA,2026-01-05,193.4,334.578124,0.658013


In [25]:
import pandas as pd
import numpy as np

DATA_PATH = "../Opsium-challenge/Data/"

forecast_df = pd.read_csv(
    DATA_PATH + "Model_forecasted_demand_output.csv"
)

capacity_df = pd.read_csv(
    DATA_PATH + "flight_capacity_master.csv"
)

print("Forecast data:", forecast_df.shape)
print("Capacity data:", capacity_df.shape)


Forecast data: (1200, 7)
Capacity data: (4, 7)


In [26]:
capacity_df = capacity_df.rename(columns={
    "fixed_cost": "fixed_operational_cost"
})


In [27]:
planning_df = forecast_df.merge(
    capacity_df,
    on="route",
    how="left"
)

planning_df.head()


,customer_id,sku_id,route,time_period,base_demand,forecasted_demand,forecast_confidence,flight_id,max_capacity,fixed_operational_cost,variable_cost_per_unit,real_time_update_flag,delay_risk_score
0,CUST1,SKU1,DEL-FRA,2026-01-01,181.4,175.217753,0.995899,FX366,2394,63576,3.558824,1,0.295745
1,CUST1,SKU1,DEL-FRA,2026-01-02,186.9,253.206883,0.792359,FX366,2394,63576,3.558824,1,0.295745
2,CUST1,SKU1,DEL-FRA,2026-01-03,229.3,1989.191211,0.687857,FX366,2394,63576,3.558824,1,0.295745
3,CUST1,SKU1,DEL-FRA,2026-01-04,197.0,363.795559,0.747558,FX366,2394,63576,3.558824,1,0.295745
4,CUST1,SKU1,DEL-FRA,2026-01-05,193.4,334.578124,0.658013,FX366,2394,63576,3.558824,1,0.295745


In [28]:
planning_df["demand_stability"] = planning_df["forecast_confidence"].apply(
    lambda x: "Stable" if x >= 0.75 else "Volatile"
)


In [29]:
planning_df["cost_profile"] = planning_df.apply(
    lambda r: "High Fixed Cost Exposure"
    if r["fixed_operational_cost"] > r["variable_cost_per_unit"] * r["max_capacity"]
    else "High Variable Cost Exposure",
    axis=1
)


In [30]:
planning_df["delay_risk_flag"] = planning_df["delay_risk_score"].apply(
    lambda x: "High Delay Risk" if x >= 0.25 else "Low Delay Risk"
)


In [31]:
planning_df["flexibility_flag"] = planning_df["real_time_update_flag"].apply(
    lambda x: "Flexible" if x == 1 else "Locked"
)


In [32]:
def decide_utilization_strategy(row):
    if row["delay_risk_flag"] == "High Delay Risk":
        return "Conservative Loading"
    
    if (
        row["demand_stability"] == "Stable"
        and row["cost_profile"] == "High Fixed Cost Exposure"
        and row["delay_risk_flag"] == "Low Delay Risk"
    ):
        return "Maximize Utilization"
    
    if (
        row["demand_stability"] == "Volatile"
        and row["flexibility_flag"] == "Flexible"
    ):
        return "Dynamic Buffer"
    
    return "Balanced Allocation"


In [33]:
planning_df["utilization_strategy"] = planning_df.apply(
    decide_utilization_strategy, axis=1
)


In [34]:
def commit_capacity(row):
    if row["utilization_strategy"] == "Maximize Utilization":
        return min(row["forecasted_demand"] * 1.05, row["max_capacity"])
    
    if row["utilization_strategy"] == "Dynamic Buffer":
        return min(row["forecasted_demand"] * 0.85, row["max_capacity"])
    
    if row["utilization_strategy"] == "Conservative Loading":
        return min(row["forecasted_demand"] * 0.70, row["max_capacity"])
    
    return min(row["forecasted_demand"] * 0.90, row["max_capacity"])


In [35]:
planning_df["committed_capacity"] = planning_df.apply(
    commit_capacity, axis=1
)


In [36]:
planning_df["load_factor"] = (
    planning_df["committed_capacity"] / planning_df["max_capacity"]
).clip(0, 1)

planning_df["void_capacity"] = (
    planning_df["max_capacity"] - planning_df["committed_capacity"]
).clip(lower=0)


In [37]:
planning_df["planning_comment"] = (
    planning_df["demand_stability"] + " | " +
    planning_df["cost_profile"] + " | " +
    planning_df["delay_risk_flag"] + " | " +
    planning_df["flexibility_flag"] + " → " +
    planning_df["utilization_strategy"]
)


In [39]:
round3_output = planning_df[[
    "route",
    "time_period",
    "forecasted_demand",
    "forecast_confidence",
    "max_capacity",
    "committed_capacity",
    "load_factor",
    "void_capacity",
    "utilization_strategy",
    "planning_comment"
]]

round3_output.to_csv(
    DATA_PATH + "model_round3_capacity_decisions.csv",
    index=False
)

round3_output.head()


,route,time_period,forecasted_demand,forecast_confidence,max_capacity,committed_capacity,load_factor,void_capacity,utilization_strategy,planning_comment
0,DEL-FRA,2026-01-01,175.217753,0.995899,2394,122.652427,0.051233,2271.347573,Conservative Loading,Stable | High Fixed Cost Exposure | High Delay...
1,DEL-FRA,2026-01-02,253.206883,0.792359,2394,177.244818,0.074037,2216.755182,Conservative Loading,Stable | High Fixed Cost Exposure | High Delay...
2,DEL-FRA,2026-01-03,1989.191211,0.687857,2394,1392.433847,0.581635,1001.566153,Conservative Loading,Volatile | High Fixed Cost Exposure | High Del...
3,DEL-FRA,2026-01-04,363.795559,0.747558,2394,254.656892,0.106373,2139.343108,Conservative Loading,Volatile | High Fixed Cost Exposure | High Del...
4,DEL-FRA,2026-01-05,334.578124,0.658013,2394,234.204687,0.097830,2159.795313,Conservative Loading,Volatile | High Fixed Cost Exposure | High Del...


In [42]:
model_round3_capacity_decisions = planning_df[[
    "route",
    "time_period",
    "forecasted_demand",
    "forecast_confidence",
    "max_capacity",
    "committed_capacity",
    "load_factor",
    "void_capacity",
    "utilization_strategy",
    "planning_comment"
]].copy()


In [43]:
def pricing_recommendation(row):
    if row["utilization_strategy"] == "Maximize Utilization":
        return "Premium / Priority Pricing"
    if row["utilization_strategy"] == "Dynamic Buffer":
        return "Flexible / Conditional Pricing"
    if row["utilization_strategy"] == "Conservative Loading":
        return "Discount / Spot Pricing"
    return "Standard Contract Pricing"

model_round3_capacity_decisions["pricing_recommendation"] = (
    model_round3_capacity_decisions.apply(pricing_recommendation, axis=1)
)


In [44]:
model_round3_capacity_decisions.to_csv(
    DATA_PATH + "model_round3_capacity_decisions.csv",
    index=False
)

model_round3_capacity_decisions.head()


,route,time_period,forecasted_demand,forecast_confidence,max_capacity,committed_capacity,load_factor,void_capacity,utilization_strategy,planning_comment,pricing_recommendation
0,DEL-FRA,2026-01-01,175.217753,0.995899,2394,122.652427,0.051233,2271.347573,Conservative Loading,Stable | High Fixed Cost Exposure | High Delay...,Discount / Spot Pricing
1,DEL-FRA,2026-01-02,253.206883,0.792359,2394,177.244818,0.074037,2216.755182,Conservative Loading,Stable | High Fixed Cost Exposure | High Delay...,Discount / Spot Pricing
2,DEL-FRA,2026-01-03,1989.191211,0.687857,2394,1392.433847,0.581635,1001.566153,Conservative Loading,Volatile | High Fixed Cost Exposure | High Del...,Discount / Spot Pricing
3,DEL-FRA,2026-01-04,363.795559,0.747558,2394,254.656892,0.106373,2139.343108,Conservative Loading,Volatile | High Fixed Cost Exposure | High Del...,Discount / Spot Pricing
4,DEL-FRA,2026-01-05,334.578124,0.658013,2394,234.204687,0.097830,2159.795313,Conservative Loading,Volatile | High Fixed Cost Exposure | High Del...,Discount / Spot Pricing
